In [1]:
import os
import math
import numpy as np
import networkx as nx
from scipy.spatial import ConvexHull
from scipy import linalg
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import ot
import porespy as ps
from skimage import draw

In [2]:
core_path = '../neuron_data/'
folders = ['Basal_ganglia', 'Cerebellum', 'Hippocampus', 'Main_olfactory_bulb', 'Retina']

label_map = {folder: idx for idx, folder in enumerate(folders)}


# Dictionaries to store data for each feature 
feature_data = {
    'N_edg': {folder: [] for folder in folders},
    'Br_l': {folder: [] for folder in folders},
    'Br_dens': {folder: [] for folder in folders},
    'Betw': {folder: [] for folder in folders},
    'G_diam': {folder: [] for folder in folders},
    'Apl': {folder: [] for folder in folders},
    'Assort': {folder: [] for folder in folders},
    'Spec_ent': {folder: [] for folder in folders},
    'Circ': {folder: [] for folder in folders},
    'q1': {folder: [] for folder in folders},
    'q2': {folder: [] for folder in folders},
    'q3': {folder: [] for folder in folders},
    'q4': {folder: [] for folder in folders},
    'Dir_std': {folder: [] for folder in folders},
    'Dir_ent': {folder: [] for folder in folders},
    'Order': {folder: [] for folder in folders},
    'FD': {folder: [] for folder in folders},
    'Alg_Conn': {folder: [] for folder in folders},
    'Energy': {folder: [] for folder in folders}
}
id_data = {folder: [] for folder in folders}



# feature_data = {
#     'number_of_egdes': {folder: [] for folder in folders},
#     'avg_branch_length': {folder: [] for folder in folders},
#     'branch_density': {folder: [] for folder in folders},
#     'mean_betweenness_centrality': {folder: [] for folder in folders},
#     'graph_diameter': {folder: [] for folder in folders},
#     'average_path_length': {folder: [] for folder in folders},
#     'assortativity': {folder: [] for folder in folders},
#     'spectral_entropy': {folder: [] for folder in folders},
#     'circuity': {folder: [] for folder in folders},
#     'mass_quartile_1': {folder: [] for folder in folders},
#     'mass_quartile_2': {folder: [] for folder in folders},
#     'mass_quartile_3': {folder: [] for folder in folders},
#     'mass_quartile_4': {folder: [] for folder in folders},
#     'directional_std_dev': {folder: [] for folder in folders},
#     'directional_entropy': {folder: [] for folder in folders},
#     'orientation_order': {folder: [] for folder in folders},
#     'fractal_dimension': {folder: [] for folder in folders},
#     'algebraic_connectivity': {folder: [] for folder in folders},
#     'bending_energy': {folder: [] for folder in folders}
# }
# id_data = {folder: [] for folder in folders}

interval_labels = [r'[0, \arccos(3/4)]', r'(\arccos(3/4), \arccos(1/2]]', r'(\arccos(1/2), \arccos(1/4]]', r'(\arccos(1/4), \pi/2]']

# Dictionary to store results
results = {folder: {} for folder in folders}

# Read the SWC file and remove axon (TypeID == 2) for Retina, optionally remove soma (TypeID == 1)
def read_swc(file_path, remove_axon=False, remove_soma=False):
    """Read an SWC file and return a DataFrame with neuron morphology data.
    If remove_axon is True, exclude axon (TypeID == 2).
    If remove_soma is True, exclude soma (TypeID == 1) and adjust parents."""
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                fields = line.split()
                if len(fields) >= 7:
                    data.append([int(fields[0]), int(fields[1]), float(fields[2]),
                                 float(fields[3]), float(fields[4]), float(fields[5]),
                                 int(fields[6])])
    columns = ['SampleID', 'TypeID', 'x', 'y', 'z', 'radius', 'parent']
    df = pd.DataFrame(data, columns=columns)
    if remove_axon:
        df = df[df['TypeID'] != 2].reset_index(drop=True)
    if remove_soma:
        df = df[df['TypeID'] != 1].reset_index(drop=True)
    # Adjust parents: if parent not in current df and != -1, set to -1
    valid_parents = set(df['SampleID'])
    df.loc[~df['parent'].isin(valid_parents) & (df['parent'] != -1), 'parent'] = -1
    return df
# Helper function to add a ball to the 3D volume
def add_ball(im, cy, cx, cz, radius):
    if radius <= 0:
        return
    r = int(np.ceil(radius))
    y_min = max(0, cy - r)
    y_max = min(im.shape[0], cy + r + 1)
    x_min = max(0, cx - r)
    x_max = min(im.shape[1], cx + r + 1)
    z_min = max(0, cz - r)
    z_max = min(im.shape[2], cz + r + 1)
    yy, xx, zz = np.meshgrid(np.arange(y_min, y_max),
                             np.arange(x_min, x_max),
                             np.arange(z_min, z_max),
                             indexing='ij')
    mask = (yy - cy)**2 + (xx - cx)**2 + (zz - cz)**2 <= radius**2
    im[yy[mask], xx[mask], zz[mask]] = True
# Create a 3D binary volume from the SWC data
def swc_to_binary_volume(swc_data, volume_size=700, radius_scale=2.0, padding_factor=1.1):
    """Convert SWC data to a 3D binary volume : voxel_size = max_extent / volume_size, dimensions proportional to lengths."""
    x_min, x_max = swc_data['x'].min(), swc_data['x'].max()
    y_min, y_max = swc_data['y'].min(), swc_data['y'].max()
    z_min, z_max = swc_data['z'].min(), swc_data['z'].max()
    delta_x = x_max - x_min
    delta_y = y_max - y_min
    delta_z = z_max - z_min
    L = max(delta_x, delta_y, delta_z)
    voxel_size = L * padding_factor / volume_size
    extent_x = delta_x * padding_factor
    extent_y = delta_y * padding_factor
    extent_z = delta_z * padding_factor
    nx = int(np.ceil(extent_x / voxel_size))
    ny = int(np.ceil(extent_y / voxel_size))
    nz = int(np.ceil(extent_z / voxel_size))
    # Cell-centered mapping: map [start, start+extent) -> indices [0..N-1] using floor
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    z_center = (z_min + z_max) / 2
    start_x = x_center - extent_x/2
    start_y = y_center - extent_y/2
    start_z = z_center - extent_z/2
    def scale_coord(coord, start, size):
        idx = int(np.floor((coord - start) / voxel_size))
        return max(0, min(idx, size - 1))
    im = np.zeros((ny, nx, nz), dtype=bool)
    for _, row in swc_data.iterrows():
        if row['TypeID'] == 1: # Soma (skipped if removed, but check anyway)
            cx = scale_coord(row['x'], start_x, nx)
            cy = scale_coord(row['y'], start_y, ny)
            cz = scale_coord(row['z'], start_z, nz)
            radius = max(1, row['radius'] / voxel_size * radius_scale)
            add_ball(im, cy, cx, cz, radius)
        elif row['parent'] != -1: # Neurites
            parent_row = swc_data[swc_data['SampleID'] == row['parent']]
            if not parent_row.empty:
                cx1 = scale_coord(row['x'], start_x, nx)
                cy1 = scale_coord(row['y'], start_y, ny)
                cz1 = scale_coord(row['z'], start_z, nz)
                cx2 = scale_coord(parent_row['x'].iloc[0], start_x, nx)
                cy2 = scale_coord(parent_row['y'].iloc[0], start_y, ny)
                cz2 = scale_coord(parent_row['z'].iloc[0], start_z, nz)
                avg_radius = (row['radius'] + parent_row['radius'].iloc[0]) / 2
                radius = max(1, avg_radius / voxel_size * radius_scale)
                # Draw line in 3D
                yy, xx, zz = draw.line_nd((cy1, cx1, cz1), (cy2, cx2, cz2))
                for y, x, z in zip(yy, xx, zz):
                    add_ball(im, y, x, z, radius)
    return im
# Function to compute fractal dimension using box-counting
def compute_fractal_dimension(swc_file, folder):
    try:
        remove_axon = (folder == 'Retina')
        swc_data = read_swc(swc_file, remove_axon=remove_axon, remove_soma=True)
        if swc_data.empty:
            return 0.0
        vol = swc_to_binary_volume(swc_data, volume_size=700, radius_scale=2.0, padding_factor=1.1)
        box_sizes = np.logspace(np.log10(5), np.log10(50), num=5, dtype=int)
        boxcount = ps.metrics.boxcount(vol, bins=box_sizes)
        stable_region = (boxcount.size >= 5) & (boxcount.size <= 50)
        if np.any(stable_region):
            return np.mean(boxcount.slope[stable_region])
        else:
            return 0.0
    except Exception as e:
        print(f"Error computing FD for {swc_file}: {e}")
        return 0.0
# Function to compute OT distance
def compute_ot_distance_1d(data1, data2):
    n_bins = max(len(data1), len(data2))
    bins = np.linspace(min(min(data1), min(data2)), max(max(data1), max(data2)), n_bins)
    hist1, _ = np.histogram(data1, bins=bins, density=True)
    hist2, _ = np.histogram(data2, bins=bins, density=True)
    hist1 = hist1 / np.sum(hist1)
    hist2 = hist2 / np.sum(hist2)
    biN_alphaenters = (bins[:-1] + bins[1:]) / 2
    M = ot.dist(biN_alphaenters.reshape(-1, 1), biN_alphaenters.reshape(-1, 1), metric='euclidean')
    return ot.emd2(hist1, hist2, M)
    
# Function to get upper triangular matrix
def upper_triangular(df):
    mask = np.triu(np.ones(df.shape), k=1).astype(bool)
    df_upper = df.where(mask)
    return df_upper
    
def average_bending_energy_curve(c, closed=False, eps=1e-12):
    """
    Compute the line integral of the squared curvature of a polyline:
        ∫ κ(s)^2 ds / len(c)
    using a discrete arclength-based scheme on edges/tangents.
    Parameters
    ----------
    c : (N, d) array_like
        Vertices of the curve (d=2 or d=3).
    closed : bool, default True
        If True, treat the curve as closed (cyclic indices).
        If False, treat as open and use only interior vertices.
    eps : float, default 1e-12
        Small clamp to avoid division by zero for tiny edges.
    Returns
    -------
    float
        Discrete approximation of ∫ κ^2 ds.
    Notes
    -----
    Let e_i = c_{i+1} - c_i, ℓ_i = ||e_i||, and t_{i+1/2} = e_i / ℓ_i.
    For a closed curve, the per-vertex contribution at vertex i is:
        ||t_{i+1/2} - t_{i-1/2}||^2 / ((ℓ_i + ℓ_{i-1})/2).
    For an open curve, the sum is taken over interior vertices only.
    """
    c = np.asarray(c, dtype=np.float64)
    if c.ndim != 2 or c.shape[1] < 2:
        raise ValueError("c must be an (N, d) array with d >= 2.")
    N, d = c.shape
    if N < 3:
        raise ValueError("Need at least 3 points to estimate curvature.")
    if closed:
        # Edges, lengths, and unit tangents on edges
        e = np.roll(c, -1, axis=0) - c # (N, d), edges i -> i+1
        ℓ = np.linalg.norm(e, axis=1)
        length = np.sum(ℓ)
        ℓ = np.maximum(ℓ, eps)
        t = e / ℓ[:, None] # unit tangents on edges
        # Δt at vertices (i): t_{i+1/2} - t_{i-1/2}
        Δt = t - np.roll(t, 1, axis=0) # (N, d)
        # Dual (Voronoi) cell length around vertex i
       
        avg_ℓ = 0.5 * (ℓ + np.roll(ℓ, 1)) # (N,)
        avg_ℓ = np.maximum(avg_ℓ, eps)
        # Sum of ||Δt||^2 / avg_ℓ
        energy = np.sum(np.einsum('ij,ij->i', Δt, Δt) / avg_ℓ)/length
        return float(energy)
    else:
        # Open curve: edges on [0..N-2]
        e = c[1:] - c[:-1] # (N-1, d)
        ℓ = np.linalg.norm(e, axis=1)
        length = np.sum(ℓ)
        ℓ = np.maximum(ℓ, eps)
        t = e / ℓ[:, None] # (N-1, d)
        # Interior vertices correspond to differences of edge tangents
        Δt = t[1:] - t[:-1] # (N-2, d)
        avg_ℓ = 0.5 * (ℓ[1:] + ℓ[:-1]) # (N-2,)
        avg_ℓ = np.maximum(avg_ℓ, eps)
        energy = np.sum(np.einsum('ij,ij->i', Δt, Δt) / avg_ℓ)/length
        return float(energy)
# Process each neuron
for folder in folders:
    folder_path = os.path.join(core_path, folder, 'CNG version')
    if not os.path.exists(folder_path):
        continue
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".swc"):
            swc_file = os.path.join(folder_path, file_name)
            points = {}
            nodes = {}
            parents = {}
            children = {}
            try:
                # Read SWC file and filter node types
                neuron_name = os.path.basename(swc_file).split('.')[0]
                valid_types = [3] if folder == 'Cerebellum' else [1, 3]
                with open(swc_file, 'r') as f:
                    for line in f:
                        if line.startswith('#') or not line.strip():
                            continue
                        parts = line.split()
                        if len(parts) == 7:
                            index, type_, x, y, z, radius, parent = map(float, parts)
                            type_ = int(type_)
                            if type_ in valid_types or (folder == 'Retina' and type_ != 2):
                                points[index] = (index, type_, x, y, z, radius, parent)
                                nodes[index] = np.array([x, y, z])
                                parents[index] = parent
                                if parent != -1 and parent in nodes:
                                    if parent not in children:
                                        children[parent] = []
                                    children[parent].append(index)
                # Identify branch points, terminals, roots, and soma nodes
                branch_points = [idx for idx in nodes if idx in children and len(children[idx]) > 1]
                terminals = [idx for idx in nodes if idx not in children]
                roots = [idx for idx in nodes if parents[idx] == -1]
                soma_nodes = [idx for idx in nodes if points[idx][1] == 1]
                # Compute branch lengths
                branch_lengths = []
                visited_segments = set()
                def compute_segment_length(start_idx, end_idx):
                    length = np.linalg.norm(nodes[start_idx] - nodes[end_idx])
                    return length
                def trace_branch(start_idx, parent_idx, current_length, visited):
                    if (start_idx, parent_idx) in visited_segments or (parent_idx, start_idx) in visited_segments:
                        return None, 0.0
                    visited_segments.add((start_idx, parent_idx))
                    visited.add(start_idx)
                    path = [start_idx]
                    total_length = current_length
                    if start_idx in terminals or parents[start_idx] == -1:
                        return path, total_length
                    if start_idx in branch_points:
                        return path, total_length
                    if start_idx in children and len(children[start_idx]) == 1:
                        child_idx = children[start_idx][0]
                        if child_idx not in visited:
                            segment_length = compute_segment_length(start_idx, child_idx)
                            new_path, new_length = trace_branch(
                                child_idx, start_idx, total_length + segment_length, visited.copy()
                            )
                            if new_path:
                                return path + new_path, new_length
                    return None, 0.0
                # Identify stem nodes (nodes connected to soma)
                stem_nodes = []
                for soma_idx in soma_nodes:
                    if soma_idx in children:
                        stem_nodes.extend([c for c in children[soma_idx] if points.get(c, [0, 0])[1] != 1])
                # Start branches from stems (or roots for Cerebellum) and bifurcation points
                start_points = stem_nodes if soma_nodes else roots
                start_points += branch_points
                # Trace branches from soma-to-stem segments
                for soma_idx in soma_nodes:
                    if soma_idx in children:
                        for child_idx in children[soma_idx]:
                            if points.get(child_idx, [0, 0])[1] == 1:
                                continue
                            segment_length = compute_segment_length(soma_idx, child_idx)
                            path, length = trace_branch(child_idx, soma_idx, segment_length, set())
                            if path and length > 0:
                                branch_lengths.append(length)
                              
                # Trace branches from other start points
                for start_idx in start_points:
                    if start_idx in children:
                        for child_idx in children[start_idx]:
                            if points.get(child_idx, [0, 0])[1] == 1:
                                continue
                            segment_length = compute_segment_length(start_idx, child_idx)
                            path, length = trace_branch(child_idx, start_idx, segment_length, set())
                            if path and length > 0:
                                branch_lengths.append(length)
                              
                # Compute branch metrics
                number_of_egdes = len(branch_lengths)
                avg_branch_length = np.mean(branch_lengths) if branch_lengths else 0
                
                # Create networkx graph for other metrics
                G = nx.Graph()
                all_nodes = {p[0]: (p[2], p[3], p[4]) for p in points.values() if p[1] in valid_types}
                edge_lengths = []
                for p in points.values():
                    if p[1] not in valid_types:
                        continue
                    start = p[0]
                    end = p[6]
                    if end != -1 and end in all_nodes:
                        sx, sy, sz = all_nodes[start]
                        ex, ey, ez = all_nodes[end]
                        weight = math.sqrt((ex - sx)**2 + (ey - sy)**2 + (ez - sz)**2)
                        G.add_edge(start, end, weight=weight)
                        edge_lengths.append(weight)
                      
                # Edge Density (modified to 3D: use volume instead of area)
                all_points = np.array([(p[2], p[3], p[4]) for p in points.values() if p[1] in valid_types])
                total_edge_length = sum(edge_lengths)
                if len(np.unique(all_points, axis=0)) >= 4: # Min 4 points for 3D hull
                    hull = ConvexHull(all_points)
                    volume = hull.volume
                else:
                    volume = 0
                branch_density = total_edge_length / volume if volume > 0 else 0
              
                # Betweenness Centrality (Mean)
                try:
                    bc = nx.betweenness_centrality(G, weight='weight', normalized=True)
                    mean_bc = np.mean(list(bc.values())) if bc else 0
                except:
                    mean_bc = 0
                # Graph graph_diameter
                try:
                    graph_diameter = nx.diameter(G, weight='weight') if nx.is_connected(G) else 0
                except:
                    graph_diameter = 0
                # Average Path Length
                try:
                    apl = nx.average_shortest_path_length(G, weight='weight') if nx.is_connected(G) else 0
                except:
                    apl = 0
                # Assortativity
                try:
                    if G.number_of_edges() < 2:
                        assortativity = 0
                    else:
                        assortativity = nx.degree_assortativity_coefficient(G, weight='weight')
                        assortativity = 0 if np.isnan(assortativity) else assortativity
                except:
                    assortativity = 0
                # Spectral Entropy
                try:
                    L = nx.laplacian_matrix(G, weight='weight').astype(float).toarray()
                    eigenvalues = linalg.eigvalsh(L)
                    eigenvalues = np.maximum(eigenvalues, 1e-12)
                    probs = eigenvalues / np.sum(eigenvalues)
                    spectral_entropy = -np.sum(probs * np.log(probs)) if probs.sum() > 0 else 0
                except:
                    spectral_entropy = 0
                # Algebraic Connectivity
                try:
                    alg_conn = nx.algebraic_connectivity(G, weight='weight', method='tracemin_pcg')
                    algebraic_connectivity = alg_conn
                except:
                    algebraic_connectivity = 0
                 
                # Circuity
                straight_line_lengths = branch_lengths # Branch lengths are 3D
                local_vectors = []
                local_lengths = []
                for idx, p in points.items():
                    if p[6] != -1 and p[6] in all_nodes:
                        dx = p[2] - points[p[6]][2]
                        dy = p[3] - points[p[6]][3]
                        dz = p[4] - points[p[6]][4]
                        length = math.sqrt(dx**2 + dy**2 + dz**2)
                        if length > 1e-10:
                            local_vectors.append((dx / length, dy / length, dz / length))
                            local_lengths.append(length)
                L_net = sum(local_lengths)
                L_g = sum(straight_line_lengths)
                circuity = L_net / L_g if L_g > 0 else 1
              
                # Fractal Dimension (modified to 3D box-counting)
                fractal_dim = compute_fractal_dimension(swc_file, folder)
              
                # Angular metrics (modified to 3D using orientation tensor)
              
                mass_quartile1 = mass_quartile2 = mass_quartile3 = mass_quartile4 = 0.0
                directional_std = 0.0
                directional_entropy = 0.0
                orientation_order = 1.0
                if local_vectors:
                    u = np.array(local_vectors) # (n_segments, 3)
                    edge_weights = np.array(local_lengths)
                    total_weight = np.sum(edge_weights)
                    edge_weights /= total_weight if total_weight > 0 else 1
                  
                    # Orientation tensor T = sum w u u^T
                    T = np.sum([w * np.outer(ui, ui) for w, ui in zip(edge_weights, u)], axis=0)
                  
                    # Eigen decomposition
                    eigvals, eigvecs = np.linalg.eigh(T)
                    lambda_max_idx = np.argmax(eigvals)
                    lambda_max = eigvals[lambda_max_idx]
                  
                    mu = eigvecs[:, lambda_max_idx] # Principal direction (mean direction)
                  
                    directional_variance = 1 - lambda_max
                  
                    directional_std = np.sqrt(directional_variance)
                   
                    """This block is for weighted entropy and orientation angles"""
                   
                  
                    # Angular deviations alpha = arccos(|u . mu|)
                    dots = np.dot(u, mu)
                    u[dots < 0] = -u[dots < 0] # Flip vectors where dot < 0
                    dots = np.abs(dots)
                    alphas = np.arccos(dots)
                    cos_alphas = dots # cos(alpha_i) = dot product (since it is unit vectors)
                    # Compute betas (azimuthal angles)
                    betas = np.zeros(len(u))
                   
                    # Canonical vector e1 = (1,0,0)^T
                   
                    e1 = np.array([1., 0., 0.])
                   
                    # Compute nu = (e1 × mu) / ||e1 × mu|| (reference direction in perpendicular plane)
                   
                    cross_e1_mu = np.cross(e1, mu)
                    norm_cross = np.linalg.norm(cross_e1_mu)
                    if norm_cross > 1e-10:
                        nu = cross_e1_mu / norm_cross
                    else:
                        # Fallback if e1 parallel to mu: use e2 = (0,1,0)
                        e2 = np.array([0., 1., 0.])
                        cross_e2_mu = np.cross(e2, mu)
                        norm_cross = np.linalg.norm(cross_e2_mu)
                        if norm_cross > 1e-10:
                            nu = cross_e2_mu / norm_cross
                        else:
                            # Rare case: mu along both, use e3
                            e3 = np.array([0., 0., 1.])
                            cross_e3_mu = np.cross(e3, mu)
                            nu = cross_e3_mu / np.linalg.norm(cross_e3_mu)
                           
                    # Second basis vector: v2 = mu × nu
                    v2 = np.cross(mu, nu)
                    # v1 = nu
                    v1 = nu
                    # For each u_i, compute projection π(u_i) = u_i - <u_i, mu> mu
                    # Then β = atan2( π(u_i) · v2, π(u_i) · v1 ) in [0, 2π)
                   
                    for i in range(len(u)):
                        inner = dots[i] # <u_i, mu>
                        proj = u[i] - inner * mu # π(u_i)
                        proj_norm = np.linalg.norm(proj)
                        if proj_norm > 1e-10:
                            beta = np.arctan2(np.dot(proj, v2), np.dot(proj, v1))
                            if beta < 0:
                                beta += 2 * np.pi
                            betas[i] = beta
                        else:
                            betas[i] = 0.0
                    # Weighted entropy using 2D histogram on cos_alpha and beta
                   
                    N_alpha = 8 # Bins for cos_alpha [0,1]
                    N_beta = 16 # Bins for beta [0, 2*pi)
                   
                    hist_weighted, xedges, yedges = np.histogram2d(
                        cos_alphas, betas, bins=(N_alpha, N_beta),
                        range=[[0, 1], [0, 2 * np.pi]], weights=edge_weights
                    )
                    total_hist = np.sum(hist_weighted)
                    if total_hist > 0:
                        p_weighted = hist_weighted / total_hist
                    else:
                        p_weighted = np.zeros_like(hist_weighted)
                       
                    # Compute entropy, avoiding log(0)
              
                    # remove all elements =0 in p_weighted to avoid log 0
              
                    p_weighted = p_weighted[p_weighted != 0]
              
                    directional_entropy = -np.sum(p_weighted * np.log(p_weighted)) # Vectorized for efficiency
                    # orientation_order (normalize by max entropy)
                   
                    H_max = np.log(N_alpha * N_beta) if N_alpha * N_beta > 0 else 0
                    orientation_order = 1 - (directional_entropy / H_max)**2 if H_max != 0 else 0
                    # Quartile masses
                    alpha1 = np.arccos(3/4) # ~0.7227 rad
                    alpha2 = np.arccos(1/2) # pi/3 ~1.0472
                    alpha3 = np.arccos(1/4) # ~1.3181
                  
                    mask_1 = alphas <= alpha1
                    mask_2 = (alphas > alpha1) & (alphas <= alpha2)
                    mask_3 = (alphas > alpha2) & (alphas <= alpha3)
                    mask_4 = alphas > alpha3
                  
                    mass_quartile1 = np.sum(edge_weights * mask_1)
                    mass_quartile2 = np.sum(edge_weights * mask_2)
                    mass_quartile3 = np.sum(edge_weights * mask_3)
                    mass_quartile4 = np.sum(edge_weights * mask_4)
                   
                # Add sum of average bending energies
                # Compute bending energies by tracing branches
                branch_avg_bendings = []
                visited_segments = set()
                def trace_branch(start_idx, parent_idx, visited):
                    """Trace a branch until a terminal, bifurcation, or root."""
                    if (start_idx, parent_idx) in visited_segments or (parent_idx, start_idx) in visited_segments:
                        return None
                    visited_segments.add((start_idx, parent_idx))
                    visited.add(start_idx)
                    path = [start_idx]
                    # Stop at terminals or roots
                    if start_idx in terminals or parents[start_idx] == -1:
                        return path
                    # Continue through single-child nodes, stop at bifurcations
                    if start_idx in children:
                        if len(children[start_idx]) > 1: # Bifurcation point
                            return path
                        # Continue through single child (continuation point)
                        child_idx = children[start_idx][0]
                        if child_idx not in visited:
                            new_path = trace_branch(
                                child_idx, start_idx, visited.copy()
                            )
                            if new_path:
                                return path + new_path
                    return None
                # Identify stem nodes (nodes connected to soma)
                stem_nodes = []
                for soma_idx in soma_nodes:
                    if soma_idx in children:
                        stem_nodes.extend(children[soma_idx])
                # Start branches from stems (for neurons with soma) or roots (for Cerebellum)
                start_points = stem_nodes if soma_nodes else roots
                start_points += branch_points # Include bifurcation points
                # Include soma-to-stem segments explicitly
                for soma_idx in soma_nodes:
                    if soma_idx in children:
                        for child_idx in children[soma_idx]:
                            if points.get(child_idx, [0, 0])[1] == 1:
                                continue
                            path = trace_branch(child_idx, soma_idx, set())
                            if path:
                                full_path = [soma_idx] + path
                                if len(full_path) >= 3:
                                    c = np.array([nodes[idx] for idx in full_path])
                                    try:
                                        avg_bend = average_bending_energy_curve(c, closed=False)
                                        branch_avg_bendings.append(avg_bend)
                                    except ValueError as e:
                                        print(f"Skipping branch in {neuron_name}: {e}")
                                        branch_avg_bendings.append(0.0)
                                else:
                                    branch_avg_bendings.append(0.0)
                # Trace branches from other start points
                for start_idx in start_points:
                    if start_idx in children:
                        for child_idx in children[start_idx]:
                            if points.get(child_idx, [0, 0])[1] == 1:
                                continue
                            path = trace_branch(child_idx, start_idx, set())
                            if path:
                                full_path = [start_idx] + path
                                if len(full_path) >= 3:
                                    c = np.array([nodes[idx] for idx in full_path])
                                    try:
                                        avg_bend = average_bending_energy_curve(c, closed=False)
                                        branch_avg_bendings.append(avg_bend)
                                    except ValueError as e:
                                        print(f"Skipping branch in {neuron_name}: {e}")
                                        branch_avg_bendings.append(0.0)
                                else:
                                    branch_avg_bendings.append(0.0)
                # Summarize bending data
                bending_energy = sum(branch_avg_bendings)
                # Store results
                neuron_name = os.path.basename(swc_file).split('.')[0]

                metrics = {


                    'N_edg': number_of_egdes,           
                    'Betw': mean_bc,
                    'Spec_ent': spectral_entropy,
                    'Alg_Conn': algebraic_connectivity,
                    'Assort': assortativity,
                    'G_diam': graph_diameter,
                    'Br_dens': branch_density,           
                    'Br_l': avg_branch_length,
                    'Energy': bending_energy,
                    'Apl': apl,
                    'Circ': circuity,
                    'FD': fractal_dim,
                    'Dir_std': directional_std,
                    'q1': mass_quartile1,
                    'q2': mass_quartile2,
                    'q3': mass_quartile3,
                    'q4': mass_quartile4,
                    'Dir_ent': directional_entropy,
                    'Order': orientation_order

                }
                
                results[folder][neuron_name] = metrics
                # Append data to feature dictionaries
                for key in feature_data:
                    feature_data[key][folder].append(metrics[key])
                id_data[folder].append(neuron_name)
            except Exception as e:
                pass
# Compile features into a DataFrame and save to CSV
data = []
for folder in folders:
    num_files = len(id_data[folder])
    for i in range(num_files):
        row = {
            'id': id_data[folder][i],
            'label': label_map[folder],
        }
        for feature in feature_data:
            row[feature] = feature_data[feature][folder][i]
        data.append(row)
df = pd.DataFrame(data)
df.to_csv('main_neuron_features.csv', index=False)



# Plot histograms for all features
              
# f, ax = plt.subplots(18, len(folders), figsize=(20, 72))
# colors = ['brown', 'purple', 'green', 'teal', 'orange', 'magenta', 'cyan', 'red', 'gold', 'darkgreen', 'blue', 'coral', 'limegreen', 'darkblue', 'pink', 'gray', 'violet', 'indigo']
# feature_titles = {
# 'number_of_egdes': 'Number of Branches',
# 'avg_branch_length': 'Average Branch Length',
# 'branch_density': 'Edge Density',
# 'mean_betweenness_centrality': 'Mean Betweenness Centrality',
# 'graph_diameter': 'Graph graph_diameter',
# 'average_path_length': 'Average Path Length',
# 'assortativity': 'Assortativity',
# 'spectral_entropy': 'Spectral Entropy',
# 'circuity': 'Circuity',
# 'mass_quartile_1': f'Mass Quartile 1: {interval_labels[0]}',
# 'mass_quartile_2': f'Mass Quartile 2: {interval_labels[1]}',
# 'mass_quartile_3': f'Mass Quartile 3: {interval_labels[2]}',
# 'mass_quartile_4': f'Mass Quartile 4: {interval_labels[3]}',
# 'directional_std_dev': 'Directional Standard Deviation',
# 'directional_entropy': 'Weighted Entropy',
# 'orientation_order': 'Orientation Order',
# 'fractal_dimension': 'Fractal Dimension',
# 'algebraic_connectivity': 'Algebraic Connectivity'
# }
# for i, (feature, data_dict) in enumerate(feature_data.items()):
# for j, folder in enumerate(folders):
# data = data_dict[folder]
# ax[i, j].clear()
# if data:
# avg_value = np.mean(data) if data else 0
# sns.histplot(data, kde=True, bins=20, color=colors[i % len(colors)], alpha=0.7, ax=ax[i, j], stat='probability')
# ax[i, j].axvline(avg_value, color='blue', linestyle='dashed', linewidth=2, label=f'Average: {avg_value:.2f}')
# ax[i, j].set_title(f'{feature_titles[feature]} in {folder}')
# ax[i, j].set_xlabel(feature_titles[feature])
# ax[i, j].set_ylabel('Frequency/Total' if j == 0 else '')
# ax[i, j].legend()
# ax[i, j].grid(axis='y', linestyle='--', alpha=0.7)
# else:
# ax[i, j].set_title(f'No data for {feature_titles[feature]} in {folder}')
# plt.suptitle('Histograms of Features per Brain Region', y=1.02)
# plt.tight_layout()
# plt.show()
# # Compute and plot OT distances for all features
# ot_results = {}
# for feature, data_dict in feature_data.items():
# ot_results[feature] = {}
# for folder1 in folders:
# ot_results[feature][folder1] = {}
# for folder2 in folders:
# data1 = data_dict[folder1]
# data2 = data_dict[folder2]
# if len(data1) > 0 and len(data2) > 0:
# dist = compute_ot_distance_1d(data1, data2)
# ot_results[feature][folder1][folder2] = dist
# df_upper = upper_triangular(pd.DataFrame(ot_results[feature]))
# df_upper = df_upper / np.max(df_upper.max()) if np.max(df_upper.max()) > 0 else df_upper
# plt.figure(figsize=(6, 4))
# sns.heatmap(df_upper, annot=True, cmap='coolwarm', fmt=".2f", mask=df_upper.isnull())
# plt.title(f"Optimal Transport Distance - {feature_titles[feature]}")
# plt.ylabel("Folder 1")
# plt.xlabel("Folder 2")
# plt.show()

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

...

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]